## Data Quality Assessment

## Objective

- The objective of this phase is to assess and improve the quality of the DataCo Supply Chain Dataset.
- This phase focuses on identifying missing values, duplicates, incorrect data types, inconsistent categories, invalid values, date issues, business-rule violations, and unusual observations before performing further analysis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv('D:\Supply Chain\Raw_Dataset.csv', encoding='latin1')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\nanga\AppData\Local\Temp\ipykernel_9424\4019962824.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv('D:\Supply Chain\Raw_Dataset.csv', encoding='latin1')


### 1. Dataset Structure and Data Types

**Objective:**  
To understand the size, structure, and data types of the dataset and identify any columns that may require further validation or correction.

#### Dataset Overview

In [8]:
# Number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# First 5 rows
display(df.head())

Rows: 180519
Columns: 53

Column Names:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'P

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,02-03-2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


#### Data Types

In [10]:
# Check data types
data_types = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notnull().sum().values
})

display(data_types)

,Column,Data Type,Non-Null Count
0,Type,object,180519
1,Days for shipping (real),int64,180519
2,Days for shipment (scheduled),int64,180519
3,Benefit per order,float64,180519
4,Sales per customer,float64,180519
5,Delivery Status,object,180519
6,Late_delivery_risk,int64,180519
7,Category Id,int64,180519
8,Category Name,object,180519
9,Customer City,object,180519


### 2. Missing Value Assessment

**Objective:**  
To identify missing values in each column and measure their proportion to understand the completeness of the dataset.

In [11]:
missing_values = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values = missing_values[missing_values['Missing Count'] > 0].sort_values('Missing Percentage', ascending= False)

display(missing_values)

,Missing Count,Missing Percentage
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


### 3. Duplicate Record Assessment

**Objective:**  
To identify duplicate records that may lead to double-counting or affect the accuracy of subsequent analysis.

In [14]:
duplicate_count = df.duplicated().sum()

print('Duplicate Count', duplicate_count)
print('Duplicate Percentage', (duplicate_count/ len(df) * 100).round(2))

Duplicate Count 0
Duplicate Percentage 0.0


### 4. Identifier Uniqueness and Consistency

**Objective:**  
To examine key identifier columns and check their uniqueness and consistency across records.

In [15]:
# Check uniqueness of important identifier columns
id_columns = [
    "Order Id",
    "Order Item Id",
    "Customer Id",
    "Product Card Id"
]

for col in id_columns:
    if col in df.columns:
        print(f"\n{col}")
        print("Total values:", df[col].count())
        print("Unique values:", df[col].nunique())
        print("Duplicate values:", df[col].duplicated().sum())


Order Id
Total values: 180519
Unique values: 65752
Duplicate values: 114767

Order Item Id
Total values: 180519
Unique values: 180519
Duplicate values: 0

Customer Id
Total values: 180519
Unique values: 20652
Duplicate values: 159867

Product Card Id
Total values: 180519
Unique values: 118
Duplicate values: 180401


### 5. Categorical Data Assessment

**Objective:**  
To review unique values and distributions of categorical columns and identify inconsistent, unexpected, or incorrectly entered categories.

In [16]:
# Check unique values and their frequencies for categorical columns
categorical_columns = df.select_dtypes(include=["object"]).columns

for col in categorical_columns:
    print(f"\n{'='*60}")
    print(f"Column: {col}")
    print("Unique values:", df[col].nunique(dropna=True))
    print(df[col].value_counts(dropna=False).head(20))


Column: Type
Unique values: 4
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

Column: Delivery Status
Unique values: 4
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Column: Category Name
Unique values: 50
Category Name
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Name: count, dtype:

### 6. Whitespace Validation

**Objective:**  
To identify leading and trailing whitespace in categorical fields that may create inconsistent categories and affect grouping or filtering.

In [17]:
# Check for leading/trailing whitespace in text columns
for col in categorical_columns:
    whitespace_count = (
        df[col].astype("string").str.strip().ne(df[col].astype("string"))
    ).sum()

    if whitespace_count > 0:
        print(f"{col}: {whitespace_count} values contain leading/trailing spaces")

Category Name: 1475 values contain leading/trailing spaces
Customer Street: 1829 values contain leading/trailing spaces
Department Name: 362 values contain leading/trailing spaces
Order Region: 17925 values contain leading/trailing spaces
Product Image: 886 values contain leading/trailing spaces
Product Name: 1774 values contain leading/trailing spaces


### 7. Numerical Data Assessment

**Objective:**  
To examine numerical columns using descriptive statistics, ranges, and distributions to identify unusual or potentially invalid values.

In [19]:
# Statistical summary of numerical columns
numeric_summary = df.describe().T

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
Benefit per order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
Sales per customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
Customer Id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
Latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933


#### Negative Values Assessment

In [20]:
# Check negative values in numerical columns
numeric_columns = df.select_dtypes(include=["number"]).columns

negative_values = {}

for col in numeric_columns:
    count = (df[col] < 0).sum()
    
    if count > 0:
        negative_values[col] = count

negative_values_df = pd.DataFrame.from_dict(
    negative_values,
    orient="index",
    columns=["Negative Value Count"]
)

negative_values_df["Negative Percentage"] = (
    negative_values_df["Negative Value Count"] / len(df) * 100
).round(2)

display(negative_values_df.sort_values(
    "Negative Value Count",
    ascending=False
))

,Negative Value Count,Negative Percentage
Longitude,180414,99.94
Benefit per order,33784,18.71
Order Item Profit Ratio,33784,18.71
Order Profit Per Order,33784,18.71
Latitude,9,0.00


#### Quantity Assessment

In [ ]:
# Check whether order quantity contains zero or negative values
if "Order Item Quantity" in df.columns:
    invalid_quantity = (df["Order Item Quantity"] <= 0).sum()

    print("Invalid quantity records:", invalid_quantity)

    print("\nQuantity distribution:")
    print(df["Order Item Quantity"].value_counts().sort_index())

Invalid quantity records: 0

Quantity distribution:
Order Item Quantity
1    99134
2    20315
3    20350
4    20335
5    20385
Name: count, dtype: int64


#### Discount Rate Assessment

In [ ]:
# Check discount rates outside the valid range 0 to 1
if "Order Item Discount Rate" in df.columns:
    invalid_discount = (
        (df["Order Item Discount Rate"] < 0) |
        (df["Order Item Discount Rate"] > 1)
    ).sum()

    print("Invalid discount rate records:", invalid_discount)

    print("\nDiscount rate summary:")
    display(df["Order Item Discount Rate"].describe())

Invalid discount rate records: 0

Discount rate summary:


count    180519.000000
mean          0.101668
std           0.070415
min           0.000000
25%           0.040000
50%           0.100000
75%           0.160000
max           0.250000
Name: Order Item Discount Rate, dtype: float64

#### Profit Ratio Assessment

In [23]:
# Check profit ratio outside the expected -1 to 1 range
if "Order Item Profit Ratio" in df.columns:
    invalid_profit_ratio = (
        (df["Order Item Profit Ratio"] < -1) |
        (df["Order Item Profit Ratio"] > 1)
    ).sum()

    print("Profit ratio outside [-1, 1]:", invalid_profit_ratio)
    
    display(df["Order Item Profit Ratio"].describe())

Profit ratio outside [-1, 1]: 6301


count    180519.000000
mean          0.120647
std           0.466796
min          -2.750000
25%           0.080000
50%           0.270000
75%           0.360000
max           0.500000
Name: Order Item Profit Ratio, dtype: float64

### 8. Date Validity and Chronological Consistency

**Objective:**  
To validate date fields and check chronological relationships between relevant dates to identify invalid or inconsistent records.

In [24]:
# Check date columns
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in date_columns:
    if col in df.columns:
        print(f"\n{col}")
        print("Data type:", df[col].dtype)
        print("Missing values:", df[col].isnull().sum())
        print("Sample values:")
        print(df[col].head())


order date (DateOrders)
Data type: object
Missing values: 0
Sample values:
0    1/31/2018 22:56
1    1/13/2018 12:27
2    1/13/2018 12:06
3    1/13/2018 11:45
4    1/13/2018 11:24
Name: order date (DateOrders), dtype: object

shipping date (DateOrders)
Data type: object
Missing values: 0
Sample values:
0    02-03-2018 22:56
1     1/18/2018 12:27
2     1/17/2018 12:06
3     1/16/2018 11:45
4     1/15/2018 11:24
Name: shipping date (DateOrders), dtype: object


#### Shipping Date vs Order Date

In [25]:
# Temporary date conversion for assessment only
order_date = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

shipping_date = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

# Check whether shipping date is before order date
invalid_date_order = (
    shipping_date < order_date
).sum()

print("Shipping date before order date:", invalid_date_order)

# Check dates that could not be interpreted
print("Invalid order dates:", order_date.isna().sum())
print("Invalid shipping dates:", shipping_date.isna().sum())

Shipping date before order date: 0
Invalid order dates: 71103
Invalid shipping dates: 108963


#### Shipping Days Assessment

In [26]:
# Check actual and scheduled shipping days
if "Days for shipping (real)" in df.columns:
    print("Actual shipping days <= 0:",
          (df["Days for shipping (real)"] <= 0).sum())

if "Days for shipment (scheduled)" in df.columns:
    print("Scheduled shipping days <= 0:",
          (df["Days for shipment (scheduled)"] <= 0).sum())

Actual shipping days <= 0: 5080
Scheduled shipping days <= 0: 9737


### 9. Business Rule Validation

**Objective:**  
To validate important business rules and logical relationships between relevant columns and identify records that violate expected conditions.

In [27]:
# Compare actual shipping days with scheduled shipping days

if (
    "Days for shipping (real)" in df.columns and
    "Days for shipment (scheduled)" in df.columns
):
    delayed_orders = (
        df["Days for shipping (real)"] >
        df["Days for shipment (scheduled)"]
    ).sum()

    on_time_or_early = (
        df["Days for shipping (real)"] <=
        df["Days for shipment (scheduled)"]
    ).sum()

    print("Orders shipped later than scheduled:", delayed_orders)
    print("Orders shipped on or before schedule:", on_time_or_early)

Orders shipped later than scheduled: 103400
Orders shipped on or before schedule: 77119


### 10. Outlier Detection

**Objective:**  
To identify potential outliers in numerical variables using the Interquartile Range (IQR) method for further review.

In [31]:
# Detect statistical outliers using the IQR method

outlier_summary = []

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()
    
    outlier_summary.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers,
        "Outlier Percentage": round(outliers / len(df) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(
    outlier_summary.sort_values(
        "Outlier Count",
        ascending=False
    )
)

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage
22,Order Profit Per Order,7.000000,64.800003,57.800003,-79.700005,151.500008,18942,10.49
2,Benefit per order,7.000000,64.800003,57.800003,-79.700005,151.500008,18942,10.49
18,Order Item Profit Ratio,0.080000,0.360000,0.280000,-0.340000,0.780000,17300,9.58
14,Order Item Discount,5.400000,29.990000,24.590000,-31.484999,66.874999,7537,4.18
27,Product Price,50.000000,199.990005,149.990005,-174.985008,424.975014,2048,1.13
17,Order Item Product Price,50.000000,199.990005,149.990005,-174.985008,424.975014,2048,1.13
3,Sales per customer,104.379997,247.399994,143.019997,-110.149998,461.929989,1943,1.08
21,Order Item Total,104.379997,247.399994,143.019997,-110.149998,461.929989,1943,1.08
10,Longitude,-98.446312,-66.370583,32.075729,-146.559906,-18.256989,1414,0.78
6,Customer Id,3258.500000,9779.000000,6520.500000,-6522.250000,19559.750000,1198,0.66


## Data Cleaning Plan

### What I Found and What I Will Do

During the Data Quality Assessment, I found some problems in the dataset. I will fix the important problems in the Data Cleaning step.

- **Missing Values:** Some columns have missing data. `Product Description` is completely empty, and `Order Zipcode` has many missing values.
  - **What I will do:** Remove `Product Description` and review whether `Order Zipcode` is useful. Handle the few missing values in other columns properly.

- **Duplicate Records:** No complete duplicate rows were found.
  - **What I will do:** No duplicate rows need to be removed.

- **Extra Spaces:** Some text columns contain unnecessary spaces before or after the values.
  - **What I will do:** Remove these extra spaces to make the text consistent.

- **Date Columns:** Order date and shipping date are stored as text instead of proper date values.
  - **What I will do:** Convert them into the correct date format and check for invalid dates.

- **Shipping Days:** Some records have unusual shipping-day values.
  - **What I will do:** Check these records and decide whether they are valid or need correction.

- **Negative Profit:** Some orders have negative profit.
  - **What I will do:** Keep these values because negative profit can mean that the company made a loss.

- **Profit Ratio:** Some records have profit-ratio values outside the expected range.
  - **What I will do:** Check these records before deciding whether they need to be corrected or removed.

- **Quantity and Discount:** No major problems were found in quantity and discount values.
  - **What I will do:** No changes are required for these columns.

- **Outliers:** Some very high or very low values were found in numerical columns.
  - **What I will do:** Check these values but will not remove them automatically because they may be real business transactions.

### Final Goal

The main goal of Data Cleaning is to fix the identified data problems and prepare a clean and reliable dataset for further analysis.